In [1]:
import numpy as np
from PIL import Image
import random


def load_image(image_path):
    """Загрузка изображения и преобразование в numpy массив"""
    img = Image.open(image_path)
    return np.array(img)


def save_image(image_array, output_path):
    """Сохранение numpy массива как изображения"""
    img = Image.fromarray(image_array.astype('uint8'))
    img.save(output_path)


def rgb_to_grayscale(rgb_image):
    """Преобразование RGB изображения в градации серого"""
    return np.dot(rgb_image[..., :3], [0.2989, 0.5870, 0.1140])


def create_adjacency_matrix(image):
    """Создание матрицы смежности для пикселей изображения"""
    height, width = image.shape
    size = height * width
    adjacency = np.zeros((size, size), dtype=np.float32)
    
    # Создаем список всех пикселей
    pixels = image.reshape(-1)
    
    # Заполняем матрицу смежности (веса ребер)
    for i in range(size):
        y1, x1 = i // width, i % width
        for j in range(i + 1, size):
            y2, x2 = j // width, j % width
            
            # Связываем только соседние пиксели (4-связность)
            if abs(x1 - x2) + abs(y1 - y2) == 1:
                # Вес ребра - разница интенсивностей
                weight = 1.0 / (1.0 + abs(float(pixels[i]) - float(pixels[j])))
                adjacency[i, j] = weight
                adjacency[j, i] = weight
                
    return adjacency


def compute_betweenness(adjacency):
    """Вычисление промежуточности для всех ребер (алгоритм Брандеса)"""
    n = adjacency.shape[0]
    betweenness = np.zeros((n, n), dtype=np.float32)
    
    for s in range(n):
        # Инициализация
        S = []
        P = [[] for _ in range(n)]
        sigma = np.zeros(n, dtype=np.float32)
        sigma[s] = 1
        d = -np.ones(n, dtype=np.int32)
        d[s] = 0
        Q = [s]
        
        # BFS
        while Q:
            v = Q.pop(0)
            S.append(v)
            for w in np.where(adjacency[v] > 0)[0]:
                if d[w] < 0:
                    Q.append(w)
                    d[w] = d[v] + 1
                if d[w] == d[v] + 1:
                    sigma[w] += sigma[v]
                    P[w].append(v)
        
        # Накопление промежуточности
        delta = np.zeros(n, dtype=np.float32)
        while S:
            w = S.pop()
            for v in P[w]:
                c = (sigma[v] / sigma[w]) * (1.0 + delta[w])
                betweenness[v, w] += c
                betweenness[w, v] += c
            if w != s:
                delta[w] += 1.0
                
    # Нормализация (так как каждое ребро учитывалось дважды)
    return betweenness / 2.0


def girvan_newman_clustering(image, num_clusters):
    """Алгоритм Гирвана-Ньюмана для кластеризации изображения"""
    # Преобразуем изображение в градации серого
    gray_image = rgb_to_grayscale(image) if image.ndim == 3 else image.copy()
    
    # Создаем матрицу смежности
    adjacency = create_adjacency_matrix(gray_image)
    
    # Копируем матрицу смежности для работы
    current_adjacency = adjacency.copy()
    
    # Начальное количество компонент связности (каждый пиксель - отдельный кластер)
    components = np.arange(adjacency.shape[0])
    
    # Пока не получим нужное количество кластеров
    while len(np.unique(components)) < num_clusters:
        # Вычисляем промежуточность
        betweenness = compute_betweenness(current_adjacency)
        
        # Находим ребро с максимальной промежуточностью
        max_betweenness = np.max(betweenness)
        edge_indices = np.where(betweenness == max_betweenness)
        i, j = edge_indices[0][0], edge_indices[1][0]
        
        # Удаляем это ребро (обнуляем вес)
        current_adjacency[i, j] = 0
        current_adjacency[j, i] = 0
        
        # Обновляем компоненты связности
        visited = np.zeros(adjacency.shape[0], dtype=bool)
        new_components = np.zeros(adjacency.shape[0], dtype=int)
        component_id = 0
        
        for node in range(adjacency.shape[0]):
            if not visited[node]:
                # BFS для определения компоненты связности
                queue = [node]
                visited[node] = True
                
                while queue:
                    current = queue.pop(0)
                    new_components[current] = component_id
                    
                    neighbors = np.where(current_adjacency[current] > 0)[0]
                    for neighbor in neighbors:
                        if not visited[neighbor]:
                            visited[neighbor] = True
                            queue.append(neighbor)
                
                component_id += 1
        
        components = new_components
    
    # Визуализация кластеров
    height, width = gray_image.shape
    clustered_image = np.zeros((height, width, 3), dtype=np.uint8)
    
    # Генерируем случайные цвета для каждого кластера
    cluster_colors = {}
    for cluster_id in np.unique(components):
        cluster_colors[cluster_id] = (
            random.randint(0, 255),
            random.randint(0, 255),
            random.randint(0, 255)
        )
    
    # Раскрашиваем пиксели в соответствии с кластерами
    for i in range(height * width):
        y, x = i // width, i % width
        clustered_image[y, x] = cluster_colors[components[i]]
    
    return clustered_image

In [2]:

input_image_path = "origins/sk.jpg"
output_image_path = "results/task_4/sk.jpg"

image = load_image(input_image_path)

# Количество кластеров (можно изменять)
num_clusters = 5

# Выполнение кластеризации
clustered_image = girvan_newman_clustering(image, num_clusters)

# Сохранение результата
save_image(clustered_image, output_image_path)
print(f"Кластеризованное изображение сохранено в {output_image_path}")

Кластеризованное изображение сохранено в results/task_4/sk.jpg


In [3]:
import numpy as np
from PIL import Image
import random

def load_image(image_path):
    """Загрузка изображения"""
    return np.array(Image.open(image_path))

def save_image(image_array, output_path):
    """Сохранение изображения"""
    Image.fromarray(image_array.astype('uint8')).save(output_path)

def rgb_to_grayscale(rgb_image):
    """Преобразование в градации серого"""
    return np.dot(rgb_image[...,:3], [0.2989, 0.5870, 0.1140])

def create_adjacency_matrix(image):
    """1. Создание матрицы смежности (аналог вычисления смежности для всех ребер)"""
    height, width = image.shape
    size = height * width
    adjacency = np.zeros((size, size), dtype=np.float32)
    
    pixels = image.reshape(-1)
    
    for i in range(size):
        y1, x1 = i // width, i % width
        for j in range(i+1, size):
            y2, x2 = j // width, j % width
            if abs(x1-x2) + abs(y1-y2) == 1:  # 4-связность
                weight = 1.0/(1.0 + abs(float(pixels[i]) - float(pixels[j])))
                adjacency[i,j] = adjacency[j,i] = weight
    return adjacency

def compute_edge_betweenness(adjacency):
    """1. Вычисление промежуточности (betweenness) для ребер (упрощенный алгоритм)"""
    n = adjacency.shape[0]
    betweenness = np.zeros((n,n), dtype=np.float32)
    
    for s in range(n):
        # Упрощенный BFS-based алгоритм вычисления betweenness
        S, P, sigma, d = [], [[] for _ in range(n)], np.zeros(n), -np.ones(n)
        sigma[s], d[s], Q = 1, 0, [s]
        
        while Q:
            v = Q.pop(0)
            S.append(v)
            for w in np.where(adjacency[v] > 0)[0]:
                if d[w] < 0:
                    Q.append(w)
                    d[w] = d[v] + 1
                if d[w] == d[v] + 1:
                    sigma[w] += sigma[v]
                    P[w].append(v)
        
        delta = np.zeros(n)
        while S:
            w = S.pop()
            for v in P[w]:
                c = sigma[v]/sigma[w] * (1 + delta[w])
                betweenness[v][w] += c
                betweenness[w][v] += c
            if w != s:
                delta[w] += 1
                
    return betweenness / 2

def girvan_newman(image, num_clusters):
    """Основной алгоритм Гирвана-Ньюмана"""
    gray_image = rgb_to_grayscale(image) if image.ndim == 3 else image.copy()
    adj = create_adjacency_matrix(gray_image)
    current_adj = adj.copy()
    components = np.arange(adj.shape[0])
    
    while len(np.unique(components)) < num_clusters:
        # 1. Вычисляем промежуточность ребер
        betweenness = compute_edge_betweenness(current_adj)
        
        # 2. Находим и удаляем ребро с максимальной промежуточностью
        max_val = np.max(betweenness)
        i, j = np.where(betweenness == max_val)
        i, j = i[0], j[0]
        current_adj[i,j] = current_adj[j,i] = 0
        
        # 3. Пересчитываем компоненты связности
        visited = np.zeros(adj.shape[0], dtype=bool)
        new_components = np.zeros(adj.shape[0], dtype=int)
        component_id = 0
        
        for node in range(adj.shape[0]):
            if not visited[node]:
                queue = [node]
                visited[node] = True
                while queue:
                    current = queue.pop(0)
                    new_components[current] = component_id
                    for neighbor in np.where(current_adj[current] > 0)[0]:
                        if not visited[neighbor]:
                            visited[neighbor] = True
                            queue.append(neighbor)
                component_id += 1
                
        components = new_components
    
    # Визуализация результата
    height, width = gray_image.shape
    result = np.zeros((height, width, 3), dtype=np.uint8)
    colors = {cid: (random.randint(0,255), random.randint(0,255), random.randint(0,255)) 
              for cid in np.unique(components)}
    
    for i in range(height*width):
        y, x = i // width, i % width
        result[y,x] = colors[components[i]]
    
    return result

In [5]:
input_image_path = "origins/sk.jpg"
output_image_path = "results/task_4/sk.jpg"

input_img = load_image(input_image_path)
output_img = girvan_newman(input_img, num_clusters=16)
save_image(output_img, output_image_path)

In [6]:
import numpy as np
from PIL import Image
import random
import math

def girvan_newman_clustering(image_path, output_path, num_clusters):
    # Загрузка изображения и преобразование в граф
    img = Image.open(image_path)
    pixels = np.array(img)
    height, width, _ = pixels.shape
    
    # Создаем граф (матрицу смежности)
    graph = create_graph_from_image(pixels)
    
    # Копируем граф для работы
    working_graph = graph.copy()
    
    # Выполняем алгоритм Гирвана-Ньюмана
    while count_connected_components(working_graph) < num_clusters:
        # Вычисляем посредничество для всех ребер
        betweenness = edge_betweenness(working_graph)
        
        # Находим ребро с максимальным посредничеством
        max_edge = None
        max_value = -1
        for edge, value in betweenness.items():
            if value > max_value:
                max_value = value
                max_edge = edge
        
        # Удаляем ребро с максимальным посредничеством
        if max_edge:
            i, j = max_edge
            working_graph[i, j] = 0
            working_graph[j, i] = 0
    
    # Находим кластеры после удаления ребер
    clusters = find_clusters(working_graph)
    
    # Раскрашиваем кластеры
    colored_pixels = color_clusters(pixels, clusters)
    
    # Сохраняем результат
    result_img = Image.fromarray(colored_pixels)
    result_img.save(output_path)

def create_graph_from_image(pixels):
    height, width, _ = pixels.shape
    size = height * width
    graph = np.zeros((size, size), dtype=np.float32)
    
    # Преобразуем изображение в граф (связываем соседние пиксели)
    for y in range(height):
        for x in range(width):
            idx = y * width + x
            # Связываем с соседями (4-связность)
            for dy, dx in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                ny, nx = y + dy, x + dx
                if 0 <= ny < height and 0 <= nx < width:
                    nidx = ny * width + nx
                    # Вес ребра - разница цветов (чем меньше разница, тем сильнее связь)
                    diff = np.linalg.norm(pixels[y, x] - pixels[ny, nx])
                    weight = 1.0 / (1.0 + diff)
                    graph[idx, nidx] = weight
                    graph[nidx, idx] = weight
    return graph

def edge_betweenness(graph):
    betweenness = {}
    nodes = range(graph.shape[0])
    
    for s in nodes:
        # Алгоритм Брандеса для вычисления посредничества
        S = []
        P = {v: [] for v in nodes}
        sigma = {v: 0 for v in nodes}
        sigma[s] = 1
        d = {v: -1 for v in nodes}
        d[s] = 0
        Q = [s]
        
        while Q:
            v = Q.pop(0)
            S.append(v)
            neighbors = [w for w in nodes if graph[v, w] > 0]
            for w in neighbors:
                if d[w] < 0:
                    Q.append(w)
                    d[w] = d[v] + 1
                if d[w] == d[v] + 1:
                    sigma[w] += sigma[v]
                    P[w].append(v)
        
        delta = {v: 0 for v in nodes}
        while S:
            w = S.pop()
            for v in P[w]:
                c = sigma[v] / sigma[w] * (1 + delta[w])
                edge = (min(v, w), max(v, w))
                betweenness[edge] = betweenness.get(edge, 0) + c
                delta[v] += c
                
    return betweenness

def count_connected_components(graph):
    visited = [False] * graph.shape[0]
    count = 0
    
    for node in range(graph.shape[0]):
        if not visited[node]:
            count += 1
            queue = [node]
            visited[node] = True
            
            while queue:
                v = queue.pop(0)
                neighbors = [w for w in range(graph.shape[0]) if graph[v, w] > 0]
                for w in neighbors:
                    if not visited[w]:
                        visited[w] = True
                        queue.append(w)
    return count

def find_clusters(graph):
    visited = [False] * graph.shape[0]
    clusters = []
    
    for node in range(graph.shape[0]):
        if not visited[node]:
            cluster = []
            queue = [node]
            visited[node] = True
            
            while queue:
                v = queue.pop(0)
                cluster.append(v)
                neighbors = [w for w in range(graph.shape[0]) if graph[v, w] > 0]
                for w in neighbors:
                    if not visited[w]:
                        visited[w] = True
                        queue.append(w)
            
            clusters.append(cluster)
    return clusters

def color_clusters(pixels, clusters):
    height, width, _ = pixels.shape
    colored = np.zeros_like(pixels)
    
    for cluster in clusters:
        # Генерируем случайный цвет для кластера
        color = np.array([random.randint(0, 255), random.randint(0, 255), random.randint(0, 255)])
        
        for node in cluster:
            y = node // width
            x = node % width
            colored[y, x] = color
    
    return colored


In [7]:
input_image_path = "origins/sk.jpg"
output_image_path = "results/task_4/sk.jpg"

girvan_newman_clustering(input_image_path, output_image_path, 5)

KeyboardInterrupt: 

In [2]:
import numpy as np
from PIL import Image
from numba import njit, prange
import time
from collections import deque

@njit
def build_adjacency_matrix(img_array):
    height, width = img_array.shape
    size = height * width
    adj_matrix = np.zeros((size, size), dtype=np.float32)
    
    for y in prange(height):
        for x in prange(width):
            node = y * width + x
            # 4-связные соседи
            if y > 0:
                neighbor = (y-1) * width + x
                diff = abs(img_array[y, x] - img_array[y-1, x])
                weight = np.exp(-(diff ** 2) / 100.0)
                adj_matrix[node, neighbor] = weight
            if y < height - 1:
                neighbor = (y+1) * width + x
                diff = abs(img_array[y, x] - img_array[y+1, x])
                weight = np.exp(-(diff ** 2) / 100.0)
                adj_matrix[node, neighbor] = weight
            if x > 0:
                neighbor = y * width + (x-1)
                diff = abs(img_array[y, x] - img_array[y, x-1])
                weight = np.exp(-(diff ** 2) / 100.0)
                adj_matrix[node, neighbor] = weight
            if x < width - 1:
                neighbor = y * width + (x+1)
                diff = abs(img_array[y, x] - img_array[y, x+1])
                weight = np.exp(-(diff ** 2) / 100.0)
                adj_matrix[node, neighbor] = weight
                
    return adj_matrix

@njit
def bfs_shortest_paths(adj_matrix, start):
    size = adj_matrix.shape[0]
    distances = np.full(size, -1, dtype=np.int32)
    counts = np.zeros(size, dtype=np.int32)
    distances[start] = 0
    counts[start] = 1
    queue = [start]
    
    while queue:
        current = queue.pop(0)
        for neighbor in range(size):
            if adj_matrix[current, neighbor] > 0:
                if distances[neighbor] == -1:
                    distances[neighbor] = distances[current] + 1
                    counts[neighbor] = counts[current]
                    queue.append(neighbor)
                elif distances[neighbor] == distances[current] + 1:
                    counts[neighbor] += counts[current]
    
    return distances, counts

@njit
def compute_edge_betweenness(adj_matrix):
    size = adj_matrix.shape[0]
    edge_betweenness = np.zeros((size, size), dtype=np.float32)
    
    for start in range(size):
        distances, counts = bfs_shortest_paths(adj_matrix, start)
        
        # Обратный проход для вычисления вклада
        nodes = np.argsort(-distances)  # Сортируем по убыванию расстояния
        node_contrib = np.zeros(size, dtype=np.float32)
        
        for node in nodes:
            if distances[node] == -1:
                continue
                
            for neighbor in range(size):
                if adj_matrix[node, neighbor] > 0 and distances[neighbor] == distances[node] - 1:
                    contrib = (node_contrib[node] + 1) * counts[neighbor] / counts[node]
                    edge_betweenness[node, neighbor] += contrib
                    edge_betweenness[neighbor, node] += contrib
                    node_contrib[neighbor] += contrib
    
    return edge_betweenness / 2  # Делим на 2 для неориентированного графа

@njit
def remove_highest_betweenness_edge(adj_matrix, betweenness):
    max_val = -1
    edge_to_remove = (-1, -1)
    
    for i in range(adj_matrix.shape[0]):
        for j in range(i+1, adj_matrix.shape[1]):
            if adj_matrix[i, j] > 0 and betweenness[i, j] > max_val:
                max_val = betweenness[i, j]
                edge_to_remove = (i, j)
    
    if edge_to_remove != (-1, -1):
        i, j = edge_to_remove
        adj_matrix[i, j] = 0
        adj_matrix[j, i] = 0
    
    return adj_matrix

@njit
def find_connected_components(adj_matrix):
    size = adj_matrix.shape[0]
    visited = np.zeros(size, dtype=np.bool_)
    components = []
    
    for node in range(size):
        if not visited[node]:
            component = []
            queue = [node]
            visited[node] = True
            
            while queue:
                current = queue.pop(0)
                component.append(current)
                
                for neighbor in range(size):
                    if adj_matrix[current, neighbor] > 0 and not visited[neighbor]:
                        visited[neighbor] = True
                        queue.append(neighbor)
            
            components.append(np.array(component, dtype=np.int32))
    
    return components

def girvan_newman_segmentation(image_path, num_clusters=2, max_iter=50, img_size=100):
    print("Загрузка изображения...")
    img = Image.open(image_path).convert('L')
    img = img.resize((img_size, img_size))
    img_array = np.array(img, dtype=np.float32)
    height, width = img_array.shape
    
    print("Построение матрицы смежности...")
    start_time = time.time()
    adj_matrix = build_adjacency_matrix(img_array)
    print(f"Матрица построена за {time.time()-start_time:.2f} сек")
    
    components = [np.arange(height * width, dtype=np.int32)]
    
    print("Запуск алгоритма Гирвана-Ньюмана...")
    iteration = 0
    
    while len(components) < num_clusters and iteration < max_iter:
        iteration += 1
        print(f"Итерация {iteration}, компонент: {len(components)}")
        start_iter = time.time()
        
        betweenness = compute_edge_betweenness(adj_matrix)
        adj_matrix = remove_highest_betweenness_edge(adj_matrix, betweenness)
        components = find_connected_components(adj_matrix)
        
        print(f"Итерация завершена за {time.time()-start_iter:.2f} сек")
    
    print("Создание сегментированного изображения...")
    output = np.zeros((height, width), dtype=np.uint8)
    for i, component in enumerate(components):
        color = int(255 * (i + 1) / max(len(components), 1))
        for node in component:
            y = node // width
            x = node % width
            output[y, x] = color
    
    return Image.fromarray(output)

In [ ]:
input_image = "origins/sk_zip.jpg"
print("Начало обработки...")
start_total = time.time()
output_image = girvan_newman_segmentation(input_image, num_clusters=10)
print(f"Общее время выполнения: {time.time()-start_total:.2f} сек")
output_image.save("output_gw.png")
output_image.show()

Начало обработки...
Загрузка изображения...
Построение матрицы смежности...
Матрица построена за 1.11 сек
Запуск алгоритма Гирвана-Ньюмана...
Итерация 1, компонент: 1


In [2]:
import numpy as np
from PIL import Image
from sklearn.feature_extraction.image import img_to_graph
from sklearn.utils.graph import connected_components
from scipy.sparse.csgraph import shortest_path
import matplotlib.pyplot as plt

def girvan_newman_image_segmentation(image_path, num_communities=2):
    """
    Реализация алгоритма Гирвана-Ньюмана для сегментации изображений
    
    Параметры:
    image_path - путь к изображению
    num_communities - желаемое количество сегментов
    """
    
    # 1. Загрузка изображения и преобразование в граф
    img = Image.open(image_path).convert('L')  # Конвертируем в оттенки серого
    img_array = np.array(img)
    
    # Создаем граф из изображения (пиксели - узлы, различия интенсивности - веса ребер)
    graph = img_to_graph(img_array)
    
    # Преобразуем веса так, чтобы большие различия означали слабые связи
    graph.data = 1 - graph.data / np.max(graph.data)
    
    # Копируем граф для работы
    working_graph = graph.copy()
    
    # 2. Итеративное удаление ребер с наибольной посреднической центральностью
    while True:
        # Вычисляем количество связанных компонентов
        n_components, labels = connected_components(working_graph, directed=False)
        
        if n_components >= num_communities:
            break
            
        # Вычисляем посредническую центральность (betweenness centrality)
        betweenness = np.zeros(working_graph.shape[0])
        
        # Для каждого узла вычисляем shortest paths (это упрощенный подход)
        for i in range(working_graph.shape[0]):
            dist_matrix = shortest_path(working_graph, directed=False, indices=i)
            betweenness += dist_matrix
            
        # Находим ребро с максимальной betweenness (упрощенно)
        max_edge = np.argmax(betweenness)
        
        # Удаляем ребро с максимальной betweenness (обнуляем вес)
        working_graph[max_edge, :] = 0
        working_graph[:, max_edge] = 0
        
    # Получаем финальные метки компонентов
    _, final_labels = connected_components(working_graph, directed=False)
    
    # Визуализация результата
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img_array, cmap='gray')
    plt.title('Исходное изображение')
    
    plt.subplot(1, 2, 2)
    plt.imshow(final_labels.reshape(img_array.shape), cmap='tab20')
    plt.title('Сегментация по сообществам')
    plt.show()
    
    return final_labels.reshape(img_array.shape)

ImportError: cannot import name 'connected_components' from 'sklearn.utils.graph' (c:\Users\Nikita\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\graph.py)

In [3]:
# Пример использования
girvan_newman_image_segmentation('origins/sk_zip.jpg', num_communities=3)

NameError: name 'girvan_newman_image_segmentation' is not defined

In [3]:
import numpy as np
from PIL import Image
from collections import defaultdict

class Edge:
    def __init__(self, u, v):
        self.u = min(u, v)
        self.v = max(u, v)
    
    def __eq__(self, other):
        return self.u == other.u and self.v == other.v
    
    def __hash__(self):
        return hash((self.u, self.v))

def girvan_newman(image_path, num_clusters=2):
    # Загрузка изображения и преобразование в градации серого
    img = Image.open(image_path).convert('L')
    img_array = np.array(img)
    height, width = img_array.shape
    
    # Создание графа (четырехсвязный)
    nodes = [(i, j) for i in range(height) for j in range(width)]
    node_index = {(i, j): idx for idx, (i, j) in enumerate(nodes)}
    num_nodes = len(nodes)
    
    # Создание списка ребер
    edges = []
    for i in range(height):
        for j in range(width):
            current_node = node_index[(i, j)]
            # Соседи (4-связность)
            neighbors = []
            if i > 0: neighbors.append((i-1, j))
            if i < height-1: neighbors.append((i+1, j))
            if j > 0: neighbors.append((i, j-1))
            if j < width-1: neighbors.append((i, j+1))
            
            for ni, nj in neighbors:
                neighbor_node = node_index[(ni, nj)]
                if current_node < neighbor_node:  # Чтобы избежать дублирования ребер
                    # Вес ребра - разница в интенсивности пикселей
                    weight = 1.0 / (1.0 + abs(int(img_array[i, j]) - int(img_array[ni, nj])))
                    edges.append((current_node, neighbor_node, weight))
    
    # Инициализация графа
    graph = defaultdict(list)
    edge_objects = []
    edge_to_index = {}
    for idx, (u, v, w) in enumerate(edges):
        edge = Edge(u, v)
        edge_objects.append((edge, w))
        edge_to_index[edge] = idx
        graph[u].append((v, idx))
        graph[v].append((u, idx))
    
    # Функция для вычисления кратчайших путей (алгоритм Дейкстры)
    def compute_shortest_paths(start):
        distances = {node: float('inf') for node in range(num_nodes)}
        distances[start] = 0
        paths = {node: [] for node in range(num_nodes)}
        paths[start] = [[start]]
        
        visited = set()
        queue = [start]
        
        while queue:
            current = queue.pop(0)
            visited.add(current)
            
            for neighbor, edge_idx in graph[current]:
                edge = edge_objects[edge_idx][0]
                weight = edge_objects[edge_idx][1]
                
                if distances[neighbor] > distances[current] + weight:
                    distances[neighbor] = distances[current] + weight
                    paths[neighbor] = [path + [neighbor] for path in paths[current]]
                elif distances[neighbor] == distances[current] + weight:
                    paths[neighbor].extend([path + [neighbor] for path in paths[current]])
            
            # Находим следующий узел с минимальным расстоянием (неоптимально, но просто)
            unvisited = {node: distances[node] for node in distances if node not in visited}
            if unvisited:
                next_node = min(unvisited, key=unvisited.get)
                queue.append(next_node)
        
        return paths
    
    # Функция для вычисления промежуточности (betweenness)
    def compute_betweenness():
        betweenness = defaultdict(float)
        
        for node in range(num_nodes):
            paths = compute_shortest_paths(node)
            
            for target in paths:
                if node == target:
                    continue
                
                all_paths = paths[target]
                if not all_paths:
                    continue
                
                # Подсчет использования ребер в кратчайших путях
                for path in all_paths:
                    for i in range(len(path)-1):
                        u = path[i]
                        v = path[i+1]
                        edge = Edge(u, v)
                        betweenness[edge] += 1.0 / len(all_paths)
        
        # Нормализация (так как каждый путь учитывался дважды)
        for edge in betweenness:
            betweenness[edge] /= 2.0
        
        return betweenness
    
    # Основной цикл алгоритма Гирвана-Ньюмана
    components = [set(range(num_nodes))]  # Начинаем с одного кластера
    
    while len(components) < num_clusters:
        # Вычисляем промежуточность
        betweenness = compute_betweenness()
        
        if not betweenness:
            break
        
        # Находим ребро с максимальной промежуточностью
        max_edge = max(betweenness, key=betweenness.get)
        max_edge_idx = edge_to_index[max_edge]
        
        # Удаляем ребро из графа
        u, v = max_edge.u, max_edge.v
        graph[u] = [(neighbor, idx) for neighbor, idx in graph[u] if idx != max_edge_idx]
        graph[v] = [(neighbor, idx) for neighbor, idx in graph[v] if idx != max_edge_idx]
        
        # Проверяем, разбился ли граф на большее количество компонент
        visited = set()
        new_components = []
        
        for node in range(num_nodes):
            if node not in visited:
                queue = [node]
                component = set()
                
                while queue:
                    current = queue.pop(0)
                    if current in visited:
                        continue
                    
                    visited.add(current)
                    component.add(current)
                    
                    for neighbor, _ in graph[current]:
                        if neighbor not in visited:
                            queue.append(neighbor)
                
                new_components.append(component)
        
        components = new_components
    
    # Создаем изображение с разными кластерами
    output = np.zeros((height, width), dtype=np.uint8)
    cluster_colors = np.linspace(0, 255, num_clusters, dtype=np.uint8)
    
    for cluster_idx, component in enumerate(components[:num_clusters]):
        for node in component:
            i, j = nodes[node]
            output[i, j] = cluster_colors[cluster_idx]
    
    return Image.fromarray(output)

In [4]:

input_image = "origins/sk_zip.jpg"  # Замените на путь к вашему изображению
output_image = girvan_newman(input_image, num_clusters=2)
output_image.save("output_gn.png")
output_image.show()

KeyboardInterrupt: 

In [3]:
import numpy as np
from PIL import Image
from collections import defaultdict
from numba import njit, types
from numba.typed import Dict, List
import heapq

# Определяем типы для Numba с явным указанием int32
int32 = types.int32
float32 = types.float32
edge_type = types.UniTuple(int32, 2)
adj_item_type = types.Tuple((int32, float32))

@njit
def compute_shortest_paths_numba(adjacency, num_nodes, start):
    distances = np.full(num_nodes, np.inf, dtype=np.float32)
    distances[start] = 0.0
    paths = [[] for _ in range(num_nodes)]
    paths[start] = [[start]]
    
    visited = np.zeros(num_nodes, dtype=np.bool_)
    heap = []
    heapq.heappush(heap, (0.0, start))
    
    while heap:
        current_dist, current = heapq.heappop(heap)
        if visited[current]:
            continue
        visited[current] = True
        
        for neighbor, weight in adjacency[current]:
            new_dist = current_dist + weight
            if new_dist < distances[neighbor]:
                distances[neighbor] = new_dist
                paths[neighbor] = [path + [neighbor] for path in paths[current]]
                heapq.heappush(heap, (new_dist, neighbor))
            elif new_dist == distances[neighbor]:
                paths[neighbor].extend([path + [neighbor] for path in paths[current]])
    
    return paths

@njit
def compute_betweenness_numba(adjacency, num_nodes, edge_indices, edge_list):
    betweenness = np.zeros(len(edge_list), dtype=np.float32)
    
    for node in range(num_nodes):
        paths = compute_shortest_paths_numba(adjacency, num_nodes, node)
        
        for target in range(num_nodes):
            if node == target:
                continue
            
            all_paths = paths[target]
            if not all_paths:
                continue
            
            path_count = len(all_paths)
            for path in all_paths:
                for i in range(len(path)-1):
                    u = path[i]
                    v = path[i+1]
                    edge = (min(u, v), max(u, v))
                    if edge in edge_indices:
                        betweenness[edge_indices[edge]] += 1.0 / path_count
    
    # Нормализация
    betweenness /= 2.0
    return betweenness

def girvan_newman_numba(image_path, num_clusters=2):
    # Загрузка изображения и преобразование в градации серого
    img = Image.open(image_path).convert('L')
    img_array = np.array(img, dtype=np.int32)  # Явно указываем int32
    height, width = img_array.shape
    
    # Создание графа (четырехсвязный)
    num_nodes = height * width
    node_index = np.arange(num_nodes, dtype=np.int32).reshape(height, width)  # int32
    
    # Создание списка ребер с Numba-совместимыми структурами
    adjacency = List()
    for _ in range(num_nodes):
        adjacency.append(List.empty_list(adj_item_type))
    
    edge_list = List()
    edge_indices = Dict.empty(key_type=edge_type, value_type=int32)
    edge_counter = 0
    
    for i in range(height):
        for j in range(width):
            current_node = node_index[i, j]
            # Соседи (4-связность)
            neighbors = []
            if i > 0: neighbors.append((i-1, j))
            if i < height-1: neighbors.append((i+1, j))
            if j > 0: neighbors.append((i, j-1))
            if j < width-1: neighbors.append((i, j+1))
            
            for ni, nj in neighbors:
                neighbor_node = node_index[ni, nj]
                if current_node < neighbor_node:  # Чтобы избежать дублирования ребер
                    # Вес ребра - разница в интенсивности пикселей
                    weight = np.float32(1.0 / (1.0 + abs(int(img_array[i, j]) - int(img_array[ni, nj]))))
                    edge = (current_node, neighbor_node)
                    edge_list.append((current_node, neighbor_node, weight))
                    edge_indices[edge] = edge_counter
                    edge_counter += 1
    
    # Заполняем adjacency list с явным приведением типов
    for u, v, w in edge_list:
        adjacency[u].append((np.int32(v), w))  # Явное приведение к int32
        adjacency[v].append((np.int32(u), w))
    
    # Основной цикл алгоритма Гирвана-Ньюмана
    components = [set(range(num_nodes))]  # Начинаем с одного кластера
    
    while len(components) < num_clusters:
        # Вычисляем промежуточность
        betweenness = compute_betweenness_numba(adjacency, num_nodes, edge_indices, edge_list)
        
        if len(betweenness) == 0:
            break
        
        # Находим ребро с максимальной промежуточностью
        max_edge_idx = np.argmax(betweenness)
        max_edge = None
        for edge, idx in edge_indices.items():
            if idx == max_edge_idx:
                max_edge = edge
                break
        
        if max_edge is None:
            break
        
        # Удаляем ребро из графа
        u, v = max_edge
        # Удаляем из adjacency list
        new_adj_u = List()
        for neighbor, weight in adjacency[u]:
            if neighbor != v:
                new_adj_u.append((neighbor, weight))
        adjacency[u] = new_adj_u
        
        new_adj_v = List()
        for neighbor, weight in adjacency[v]:
            if neighbor != u:
                new_adj_v.append((neighbor, weight))
        adjacency[v] = new_adj_v
        
        # Проверяем, разбился ли граф на большее количество компонент
        visited = set()
        new_components = []
        
        for node in range(num_nodes):
            if node not in visited:
                queue = [node]
                component = set()
                
                while queue:
                    current = queue.pop(0)
                    if current in visited:
                        continue
                    
                    visited.add(current)
                    component.add(current)
                    
                    for neighbor, _ in adjacency[current]:
                        if neighbor not in visited:
                            queue.append(neighbor)
                
                new_components.append(component)
        
        components = new_components
    
    # Создаем изображение с разными кластерами
    output = np.zeros((height, width), dtype=np.uint8)
    cluster_colors = np.linspace(0, 255, num_clusters, dtype=np.uint8)
    
    for cluster_idx, component in enumerate(components[:num_clusters]):
        for node in component:
            i = node // width
            j = node % width
            output[i, j] = cluster_colors[cluster_idx]
    
    return Image.fromarray(output)

In [4]:

input_image = "origins/sk_zip.jpg"  # Замените на путь к вашему изображению
output_image = girvan_newman_numba(input_image, num_clusters=2)
output_image.save("output_numba.png")
output_image.show()

c:\Prog_tasks\CV\env\Lib\site-packages\numba\typed\typedlist.py:82: NumbaTypeSafetyWarning: unsafe cast from Tuple(int32, float64) to Tuple(int32, float32). Precision may be lost.
  l.append(item)


TypingError: Failed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1m[1m[1mFailed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1mNo implementation of function Function(<built-in function setitem>) found for signature:
 
 >>> setitem(list(undefined)<iv=None>, int64, list(list(int64)<iv=None>)<iv=None>)
 
There are 16 candidate implementations:
[1m    - Of which 14 did not match due to:
    Overload of function 'setitem': File: <numerous>: Line N/A.
      With argument(s): '(list(undefined)<iv=None>, int64, list(list(int64)<iv=None>)<iv=None>)':[0m
[1m     No match.[0m
[1m    - Of which 2 did not match due to:
    Overload in function 'SetItemSequence.generic': File: numba\core\typing\collections.py: Line 56.
      With argument(s): '(list(undefined)<iv=None>, int64, list(list(int64)<iv=None>)<iv=None>)':[0m
[1m     Rejected as the implementation raised a specific error:
       TypingError: [1minvalid setitem with value of list(list(int64)<iv=Non

In [5]:
import numpy as np
from PIL import Image
from collections import deque

def girvan_newman_algorithm(image_path, max_size=50, max_iterations=100):
    # Загрузка изображения и конвертация в оттенки серого
    image = Image.open(image_path).convert('L')
    
    # Изменение размера для вычислительной эффективности
    if image.width > max_size or image.height > max_size:
        scale = min(max_size / image.width, max_size / image.height)
        new_size = (int(image.width * scale), int(image.height * scale))
        image = image.resize(new_size)
    
    # Преобразование в массив numpy
    img_array = np.array(image)
    rows, cols = img_array.shape
    
    # Создание графа - каждый пиксель это узел, связанный с соседями
    graph = {}
    
    # Добавление узлов и рёбер (четырёхсвязный граф)
    for i in range(rows):
        for j in range(cols):
            node = (i, j)
            neighbors = []
            
            # Четыре соседа (верх, низ, лево, право)
            for ni, nj in [(i-1, j), (i+1, j), (i, j-1), (i, j+1)]:
                if 0 <= ni < rows and 0 <= nj < cols:
                    neighbors.append((ni, nj))
            
            graph[node] = neighbors
    
    # Шаг 1: Вычисление промежуточности для всех рёбер графа
    def calculate_betweenness():
        betweenness = {}
        
        # Для каждого исходного узла
        for source in graph:
            # BFS для поиска кратчайших путей
            distances = {node: float('inf') for node in graph}
            distances[source] = 0
            
            # Количество кратчайших путей
            num_paths = {node: 0 for node in graph}
            num_paths[source] = 1
            
            # Очередь для BFS
            queue = deque([source])
            
            # Предшественники на кратчайших путях
            predecessors = {node: [] for node in graph}
            
            # BFS для поиска всех кратчайших путей
            while queue:
                node = queue.popleft()
                
                for neighbor in graph[node]:
                    # Новый кратчайший путь
                    if distances[neighbor] == float('inf'):
                        distances[neighbor] = distances[node] + 1
                        num_paths[neighbor] = num_paths[node]
                        predecessors[neighbor] = [node]
                        queue.append(neighbor)
                    # Ещё один кратчайший путь
                    elif distances[neighbor] == distances[node] + 1:
                        num_paths[neighbor] += num_paths[node]
                        predecessors[neighbor].append(node)
            
            # Расчёт зависимости и промежуточности
            dependency = {node: 0 for node in graph}
            
            # Обработка узлов в порядке убывания расстояния от источника
            for node in sorted(graph, key=lambda x: distances.get(x, float('inf')), reverse=True):
                if node == source:
                    continue
                
                for predecessor in predecessors[node]:
                    # Расчёт веса
                    weight = num_paths[predecessor] / num_paths[node] * (1 + dependency[node])
                    dependency[predecessor] += weight
                    
                    # Добавление к промежуточности ребра
                    edge = tuple(sorted([predecessor, node]))
                    if edge not in betweenness:
                        betweenness[edge] = 0
                    betweenness[edge] += weight
        
        return betweenness
    
    # Функция для нахождения сообществ (связных компонент)
    def find_communities():
        visited = set()
        communities = []
        
        for node in graph:
            if node not in visited:
                community = []
                queue = deque([node])
                visited.add(node)
                
                while queue:
                    current = queue.popleft()
                    community.append(current)
                    
                    for neighbor in graph[current]:
                        if neighbor not in visited:
                            visited.add(neighbor)
                            queue.append(neighbor)
                
                communities.append(community)
        
        return communities
    
    # Основной алгоритм Гирвана-Ньюмана
    communities_history = []
    
    # Начальные сообщества
    initial_communities = find_communities()
    communities_history.append(initial_communities)
    
    # Продолжаем до max_iterations или пока не закончатся рёбра
    for _ in range(min(max_iterations, rows * cols)):
        # Шаг 1: Вычисляем промежуточность
        edge_betweenness = calculate_betweenness()
        
        if not edge_betweenness:
            break
        
        # Шаг 2: Удаляем ребро с наивысшим показателем
        max_edge = max(edge_betweenness.items(), key=lambda x: x[1])[0]
        node1, node2 = max_edge
        
        # Удаление ребра
        if node2 in graph[node1]:
            graph[node1].remove(node2)
        if node1 in graph[node2]:
            graph[node2].remove(node1)
        
        # Шаг 3: Пересчитываем величину смежности с учетом изменений 
        # (это происходит на следующей итерации в calculate_betweenness())
        
        # Шаг 4: Находим новые сообщества и повторяем процесс
        current_communities = find_communities()
        communities_history.append(current_communities)
        
        # Останавливаемся, если все узлы изолированы
        if len(current_communities) == rows * cols:
            break
    
    return communities_history, (cols, rows)

# Функция для визуализации сообществ
def visualize_communities(communities, size):
    width, height = size
    result = np.zeros((height, width, 3), dtype=np.uint8)
    
    # Присваиваем разные цвета каждому сообществу
    for i, community in enumerate(communities):
        # Генерируем случайный цвет
        color = np.random.randint(0, 256, 3)
        
        # Окрашиваем все пиксели в этом сообществе
        for node in community:
            x, y = node
            if 0 <= x < height and 0 <= y < width:
                result[x, y] = color
    
    # Преобразуем в изображение
    result_image = Image.fromarray(result)
    return result_image


In [ ]:
communities_history, size = girvan_newman_algorithm("origins/sk.jpg")
# Визуализация последнего набора сообществ
result_image = visualize_communities(communities_history[-1], size)
result_image.save("result/task_4/communities.jpg")